Train the 5-seed ResNet-18 ensemble on Kaggle (GPU T4). Inputs: the cifar10-python dataset. Each seed takes about 25 minutes, and the checkpoints are saved to /kaggle/working/outputs/checkpoints.

In [1]:
!nvidia-smi

Thu Nov 19 20:44:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /tmp
![ -d cifar10c-calibration ] || git clone https://github.com/TiaraAnnaJain/cifar10c-calibration.git
%cd /tmp/cifar10c-calibration
!git pull
!pip install -q uv
!uv sync --locked

/tmp
Cloning into 'cifar10c-calibration'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 156 (delta 68), reused 136 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 105.68 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/tmp/cifar10c-calibration
Already up to date.
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 54 packages in 1ms
⠙ Preparing packages... (0/53)                                                  warning: `build_system.requires = ["uv-build>=0.10.12,<0.11"]` does not contain the current uv version 0.13.9
Prepared 53 packages in 6m 06s                                           
Installed 53 packages in 366ms                              
 + ast-serialize==0.13.1
 + cifar10c-calibration==0.1.0 (from file:///tmp/cifar10c-calibration)
 + contourpy==1.4.0
 + cuda-bindings==13.5.0
 + cu

In [3]:
!mkdir -p /tmp/data/cifar-10-batches-py
!cp $(dirname $(find /kaggle/input -name data_batch_1 | head -1))/* /tmp/data/cifar-10-batches-py/
!ls /tmp/data/cifar-10-batches-py

batches.meta  data_batch_2  data_batch_4  readme.html
data_batch_1  data_batch_3  data_batch_5  test_batch


In [4]:
check = !uv run python -c "import torch; print(torch.__version__, torch.version.cuda); print(torch.cuda.is_available())"
print(*check, sep="\n")
assert check[-1] == "True", "PyTorch can't see the GPU, so stop here"

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
2.15.0+cu132 13.2
True


In [5]:
!uv run python scripts/train.py --fast-dev-run --data-dir /tmp/data --output-dir /kaggle/working/outputs

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
seed 0 | device cuda | epochs 1
epoch   1/1  loss 2.7639  acc 0.1055
done in 0.0 min | val acc 0.1055 | test acc 0.1055
saved /kaggle/working/outputs/fast_dev_run/checkpoints/resnet18_seed0.pt and resnet18_seed0.json


In [6]:
%cd /tmp/cifar10c-calibration
for seed in range(5):
    !uv run python scripts/train.py --seed {seed} --data-dir /tmp/data --output-dir /kaggle/working/outputs

/tmp/cifar10c-calibration
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
seed 0 | device cuda | epochs 40
epoch   1/40  loss 1.8502  acc 0.3378
epoch   2/40  loss 1.3225  acc 0.5186
epoch   3/40  loss 1.0388  acc 0.6293
epoch   4/40  loss 0.8280  acc 0.7083
epoch   5/40  loss 0.6930  acc 0.7593
epoch   6/40  loss 0.6036  acc 0.7914
epoch   7/40  loss 0.5518  acc 0.8108
epoch   8/40  loss 0.5063  acc 0.8269
epoch   9/40  loss 0.4772  acc 0.8372
epoch  10/40  loss 0.4519  acc 0.8454
epoch  11/40  loss 0.4422  acc 0.8476
epoch  12/40  loss 0.4140  acc 0.8587
epoch  13/40  loss 0.4053  acc 0.8598
epoch  14/40  loss 0.3803  acc 0.8709
epoch  15/40  loss 0.3688  acc 0.8745
epoch  16/40  loss 0.3454  acc 0.8820
epoch  17/40  loss 0.3335  acc 0.8856
epoch  18/40  loss 0.3131  acc 0.8928
epoch  19/40  loss 0.2964  acc 0.8996
epoch  20/40  loss 0.2783  acc 0.9042
epoch  21/40  loss 0.2654  acc 0.9086
epoch  22/40  loss 0.2450  acc 0.9165
epo